# Домашнее задание 21. Честный ML workflow на kNN: категориальные признаки

**Тема.** Классификация методом k ближайших соседей на таблице с **категориальными** признаками, пропусками и «опасными» колонками.

**Цель задания** — не «получить красивую метрику», а пройти весь цикл **без утечек**: от постановки задачи до единственной проверки на test. Метрика важна ровно настолько, насколько честно она получена.

**Что понадобится из урока:** train / validation / test, baseline, `fit` только на train, журнал экспериментов, recall / precision, confusion matrix.

**Как сдавать.** Ноутбук целиком (код + ответы в markdown-ячейках «**Ответ:**»). Перед сдачей: `Kernel → Restart & Run All` — всё должно выполняться сверху вниз без ошибок.

## Легенда и данные

Телеком-оператор хочет заранее находить клиентов, которые **уйдут в ближайший месяц** (отток, churn), и предлагать им удерживающую скидку. Бюджет скидок ограничен, поэтому звонить всем подряд нельзя.

Датасет **синтетический** (генерируется ячейкой ниже, один и тот же у всех). Одна строка — один клиент на момент прогноза.

| Колонка | Тип | Комментарий |
|---|---|---|
| `customer_id` | идентификатор | уникальный номер клиента |
| `age` | число | возраст, есть пропуски |
| `tenure_months` | число | сколько месяцев клиент с нами |
| `monthly_charge` | число | ежемесячный платёж, есть пропуски |
| `support_calls` | число | звонков в поддержку за последние 3 месяца |
| `contract` | категория | `Month-to-month` / `One year` / `Two year` |
| `plan_tier` | категория | `Basic` / `Standard` / `Premium` |
| `internet_service` | категория | `DSL` / `Fiber` / `None` |
| `payment_method` | категория | способ оплаты, есть пропуски и **редкая** категория |
| `region` | категория | регион |
| `has_partner` | категория | `yes` / `no` |
| `cancel_reason` | категория | причина расторжения договора, записывается при оформлении ухода |
| `churn` | **цель** | 1 — клиент ушёл, 0 — остался |

In [ ]:
# Ячейку не редактировать: она создаёт один и тот же датасет у всех.
import numpy as np
import pandas as pd

def make_churn_data(n=2000, seed=2026):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 76, n).astype(float)
    tenure = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    contract = rng.choice(['Month-to-month', 'One year', 'Two year'], n, p=[.55, .25, .20])
    plan = rng.choice(['Basic', 'Standard', 'Premium'], n, p=[.40, .40, .20])
    internet = rng.choice(['DSL', 'Fiber', 'None'], n, p=[.35, .45, .20])
    payment = rng.choice(['card', 'bank_transfer', 'e_wallet', 'cash', 'crypto'], n,
                         p=[.35, .30, .245, .10, .005])
    region = rng.choice(['North', 'South', 'East', 'West', 'Center'], n)
    partner = rng.choice(['yes', 'no'], n, p=[.48, .52])

    plan_price = pd.Series(plan).map({'Basic': 20, 'Standard': 40, 'Premium': 65}).to_numpy()
    net_price = pd.Series(internet).map({'None': 0, 'DSL': 15, 'Fiber': 30}).to_numpy()
    monthly = (plan_price + net_price + rng.normal(0, 6, n)).round(2)
    support_calls = rng.poisson(0.8 + 0.9 * (internet == 'Fiber') + 0.5 * (plan == 'Basic'), n)

    logit = (-2.3
             + 2.2 * (contract == 'Month-to-month') - 1.2 * (contract == 'Two year')
             - 0.04 * tenure
             + 0.03 * (monthly - 60)
             + 0.5 * support_calls
             + 0.9 * (internet == 'Fiber')
             + 0.9 * (payment == 'e_wallet')
             - 0.4 * (partner == 'yes')
             + 0.012 * (45 - age)
             + rng.normal(0, 0.3, n))
    churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    reason = np.where(churn == 1,
                      rng.choice(['price', 'competitor', 'service', 'moved'], n), 'none')

    df = pd.DataFrame({
        'customer_id': [f'C{i:05d}' for i in range(n)],
        'age': age, 'tenure_months': tenure, 'monthly_charge': monthly,
        'support_calls': support_calls,
        'contract': contract, 'plan_tier': plan, 'internet_service': internet,
        'payment_method': payment, 'region': region, 'has_partner': partner,
        'cancel_reason': reason, 'churn': churn,
    })
    df.loc[rng.random(n) < 0.05, 'age'] = np.nan
    df.loc[rng.random(n) < 0.03, 'monthly_charge'] = np.nan
    df.loc[rng.random(n) < 0.04, 'payment_method'] = np.nan
    return df

df = make_churn_data()
df.head()

## Алгоритм честного ML workflow

Это «закон» вашей работы. Каждый шаг ниже соответствует заданию в ноутбуке. Правило, которое проверяет весь алгоритм: **test не участвует ни в одном решении до самого последнего шага.**

| № | Шаг | Что делаем | Честное правило |
|---|---|---|---|
| 0 | **Постановка** | Фиксируем объект, момент прогноза, цель, действие, главную метрику и **критерий успеха** | Пишем **до** просмотра любых результатов. Планку потом не двигаем |
| 1 | **Аудит данных** | Типы, пропуски, категории, редкие значения, идентификаторы, колонки «из будущего» | Исключаем колонки по **смыслу** (момент прогноза), а не потому что «так лучше по метрике» |
| 2 | **Split** | `train / validation / test` = 60 / 20 / 20, `stratify`, фиксированный `random_state` | Делим **до** подсчёта любых статистик. Способ деления повторяет то, что будет «новым» после внедрения. Test «убираем в сейф» |
| 3 | **Baseline** | `DummyClassifier` на validation | Любую модель сравниваем с ним. Не лучше baseline — рано усложнять |
| 4 | **Предобработка** | Импутация пропусков → кодирование категорий → масштабирование, всё в одном `Pipeline` / `ColumnTransformer` | `fit` — **только на train**, `transform` — на train, validation и test теми же параметрами |
| 5 | **Эксперименты** | Сравниваем гипотезы о признаках и кодировании, ведём **журнал** | Один split, одна метрика, **одно** изменение за раз |
| 6 | **Подбор гиперпараметра** | Перебираем `k` | Только по **validation**, по главной метрике задачи и с контрольной (guard) метрикой |
| 7 | **Проверка критерия** | Сверяем лучший вариант с критерием из шага 0 | Решение «годится / не годится» принимается **на validation**, до test |
| 8 | **Анализ ошибок** | Confusion matrix, ошибки по группам | Смотрим на validation. Выводы о признаках — гипотезы для следующего цикла, а не повод подсмотреть test |
| 9 | **Финал** | Фиксируем одну конфигурацию и **один раз** считаем метрики на test | После просмотра test ничего не меняем. Плохой результат — это результат, его честно описывают |
| 10 | **Выводы и внедрение** | Итоговый вывод, риски, мониторинг | Что мониторить после запуска: дрейф данных и качество |

**Шпаргалка «что где можно»**

| Действие | Train | Validation | Test |
|---|:---:|:---:|:---:|
| `fit` импутера / энкодера / scaler / модели | ✅ | ❌ | ❌ |
| `transform` / `predict` | ✅ | ✅ | ✅ (только финал) |
| Выбор признаков, кодирования, `k` | ❌ | ✅ | ❌ |
| Финальная оценка выбранного решения | ❌ | ❌ | ✅ **один раз** |

## Задание 0. Постановка задачи (шаг 0)

**До того как смотреть на данные**, заполните таблицу постановки и **критерий успеха**.

Критерий успеха уже задан заказчиком — перепишите его в таблицу и не меняйте:

> На validation: **recall класса «уйдёт» ≥ 0,50** и при этом **precision ≥ 0,50**.

Подсказка: звонок с предложением скидки стоит денег, но потерянный клиент дороже. Подумайте, почему нужна **пара** метрик, а не одна.

**Ответ:**

| Пункт | Ваш ответ |
|---|---|
| объект | |
| момент прогноза | |
| цель (что предсказываем, какие классы) | |
| действие по прогнозу | |
| главная метрика (и `pos_label`) | |
| контрольная (guard) метрика | |
| критерий успеха | |

Почему одного recall недостаточно? (1–2 предложения)

## Задание 1. Аудит данных (шаг 1)

Выполните ячейку и ответьте на вопросы. **Никаких преобразований данных пока не делаем** — только смотрим.

In [ ]:
# 1) размер, типы, доля классов
print('Размер:', df.shape)
print(df.dtypes)
print('Доля класса 1 (churn):', ...)

# 2) пропуски по колонкам
...

# 3) уникальные значения и частоты категориальных признаков (включая NaN)
cat_cols = ['contract', 'plan_tier', 'internet_service', 'payment_method', 'region', 'has_partner', 'cancel_reason']
for c in cat_cols:
    ...

# 4) сколько уникальных customer_id? Совпадает ли с числом строк?
...

# 5) как cancel_reason связана с churn? (таблица сопряжённости)
...

**Ответ:**

1. Какие колонки вы **не** будете использовать как признаки и **почему**? (для каждой — аргумент через «момент прогноза» или «смысл», а не через качество модели)
2. Какие категориальные признаки **порядковые** (есть естественный порядок), а какие **номинальные**?
3. Какие проблемы данных вы нашли (пропуски, редкие категории)? Что может сломаться, если редкая категория окажется только в validation или test?
4. Объекты независимы (одна строка = один клиент). Какой способ разбиения подходит и почему?

## Задание 2. Разбиение train / validation / test (шаг 2)

1. Соберите матрицу признаков `X` — **без** `customer_id`, `cancel_reason` и `churn` (решение из задания 1) — и вектор цели `y`.
2. Разбейте данные **60 / 20 / 20** с `stratify` и `random_state=42`. Подсказка: два вызова `train_test_split`, как на уроке (сначала отложить test 20 %, потом из остатка взять validation: `0.25` от 80 % = 20 % от всех).
3. Проверьте: размеры, долю класса 1 в каждой части, отсутствие пересечения индексов.

**С этого момента test не трогаем до задания 9.**

In [ ]:
from sklearn.model_selection import train_test_split

drop_cols = [...]            # какие колонки убираем из признаков
X = ...
y = ...

X_train_val, X_test, y_train_val, y_test = ...
X_train, X_val, y_train, y_val = ...

print('train / val / test:', len(X_train), len(X_val), len(X_test))
for name, part in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(name, 'доля класса 1:', ...)

# проверка отсутствия пересечений
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0

**Ответ:** Зачем делить именно до заполнения пропусков и масштабирования? Что произойдёт, если сначала посчитать медиану возраста по всей таблице?

## Задание 3. Baseline (шаг 3)

Обучите `DummyClassifier(strategy='most_frequent')` на train, посчитайте на **validation**: accuracy, recall (`pos_label=1`) и precision (`pos_label=1`, с `zero_division=0`).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

dummy = ...
dummy.fit(X_train, y_train)
dummy_pred = ...

print('Baseline accuracy :', ...)
print('Baseline recall   :', ...)
print('Baseline precision:', ...)

**Ответ:** Почему у baseline высокая accuracy, но нулевой recall? Что это говорит о выборе метрики в задаче с редким классом?

## Задание 4. Предобработка как единый pipeline (шаг 4)

kNN считает расстояния между числами, поэтому каждый столбец должен стать **числом без пропусков и в сопоставимом масштабе**. Соберите функцию `make_preprocessor`, которая возвращает `ColumnTransformer` с тремя ветками:

| Ветка | Колонки | Шаги |
|---|---|---|
| числовые | `num_cols` | `SimpleImputer(strategy='median')` → `StandardScaler` (если `scale=True`) |
| порядковые | `ord_cols` | `SimpleImputer(strategy='most_frequent')` → `OrdinalEncoder` с **явным порядком** категорий → `StandardScaler` (если `scale=True`) |
| номинальные | `nom_cols` | `SimpleImputer(strategy='constant', fill_value='unknown')` → `OneHotEncoder(handle_unknown='ignore')` |

Порядки категорий уже заданы в словаре `ORDER`. Если какого-то списка колонок нет (пустой список) — соответствующую ветку не добавляем.

Затем:
1. Сделайте `fit` на **train**, `transform` на train и validation; выведите размерности.
2. Убедитесь, что `fit` был на train: среднее масштабированного `age` на train ≈ 0, а на validation — **нет** (не равно нулю точно). Почему?
3. Убедитесь, что `transform(X_val)` не падает, даже если в validation есть категория, которой нет в train (проверьте на искусственной строке с `payment_method='barter'`).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

NUM_COLS = ['age', 'tenure_months', 'monthly_charge', 'support_calls']
ORD_COLS = ['plan_tier', 'contract']
NOM_COLS = ['internet_service', 'payment_method', 'region', 'has_partner']
ORDER = {
    'plan_tier': ['Basic', 'Standard', 'Premium'],
    'contract': ['Month-to-month', 'One year', 'Two year'],
}

def make_preprocessor(num_cols, ord_cols, nom_cols, scale=True):
    transformers = []
    if num_cols:
        steps = [('imputer', ...)]
        if scale:
            steps.append(('scaler', ...))
        transformers.append(('num', Pipeline(steps), num_cols))
    if ord_cols:
        ...
    if nom_cols:
        ...
    return ColumnTransformer(transformers)

prep = make_preprocessor(NUM_COLS, ORD_COLS, NOM_COLS)
Z_train = prep.fit_transform(X_train)   # fit ТОЛЬКО на train
Z_val = prep.transform(X_val)
print('train:', Z_train.shape, '| val:', Z_val.shape)

# 2) среднее первого столбца (age после импутации и scaler) на train и на val
print('mean(age) train:', ...)
print('mean(age) val  :', ...)

# 3) неизвестная категория не должна ронять transform
row = X_val.iloc[[0]].copy()
row['payment_method'] = 'barter'
print('transform с неизвестной категорией — shape:', ...)

**Ответ:**
1. Почему среднее `age` на validation не равно нулю — это баг или норма?
2. Почему для `plan_tier` и `contract` мы задаём порядок вручную, а `region` кодируем one-hot? Что «соврёт» расстояние kNN, если закодировать `region` числами 0, 1, 2, 3, 4?
3. Зачем `handle_unknown='ignore'` и что при этом произойдёт с неизвестной категорией?

## Задание 5. Журнал экспериментов (шаг 5)

Напишите функцию `run_experiment(name, num_cols, ord_cols, nom_cols, scale, k)`, которая:
1. строит `Pipeline([('prep', make_preprocessor(...)), ('knn', KNeighborsClassifier(n_neighbors=k))])`;
2. делает `fit` на train;
3. считает на **validation** accuracy, recall и precision класса 1;
4. возвращает словарь-строку журнала.

Проведите гипотезы при **одинаковом `k=5`** и одном и том же split:

| ID | Гипотеза | Что меняем относительно предыдущей |
|---|---|---|
| H1 | только числовые признаки | — |
| H2 | + порядковые (ordinal) и номинальные (one-hot) | добавили категории |
| H3 | `plan_tier` и `contract` кодируем **one-hot** вместо ordinal | способ кодирования порядковых |
| H4 | как H2, но **без масштабирования** | убрали scaler |

Добавьте в таблицу строку baseline из задания 3. Выведите журнал датафреймом.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def run_experiment(name, num_cols, ord_cols, nom_cols, scale, k):
    model = Pipeline([
        ('prep', make_preprocessor(num_cols, ord_cols, nom_cols, scale=scale)),
        ('knn', KNeighborsClassifier(n_neighbors=k)),
    ])
    model.fit(X_train, y_train)
    pred = ...
    return {
        'эксперимент': name, 'k': k,
        'val_acc': ..., 'val_recall_1': ..., 'val_precision_1': ...,
    }

journal = [{
    'эксперимент': 'Baseline (most_frequent)', 'k': None,
    'val_acc': ..., 'val_recall_1': ..., 'val_precision_1': ...,
}]
journal.append(run_experiment('H1: только числа', NUM_COLS, [], [], True, 5))
# H2, H3, H4 — допишите по таблице

journal_df = pd.DataFrame(journal)
journal_df

**Ответ:**
1. Что дало добавление категориальных признаков (H1 → H2)?
2. Какой способ кодирования `plan_tier` / `contract` лучше (H2 vs H3) и можно ли делать надёжный вывод по такой разнице? Что бы вы сделали, чтобы проверить, что разница не случайна?
3. Почему kNN без масштабирования (H4) хуже? Какой признак «перетягивает» расстояние?
4. Почему в журнале меняется **одна** вещь за раз?

## Задание 6. Подбор `k` по validation (шаг 6)

Возьмите лучшую по журналу конфигурацию признаков (в эталонном решении это H2; если ваш журнал показал другое — выбирайте по своему журналу, но обоснуйте).

1. Переберите `k ∈ [1, 3, 5, 7, 11, 21, 41]`, посчитайте recall и precision класса 1 на validation.
2. Постройте график: по оси X — `k`, по Y — recall и precision; горизонтальная линия — порог 0,50.
3. **Правило выбора `k` (зафиксировано заранее):** среди `k`, у которых `precision ≥ 0,50`, берём `k` с максимальным recall; при равенстве — больший `k` (более гладкая модель). Реализуйте правило кодом.

In [ ]:
import matplotlib.pyplot as plt

K_GRID = [1, 3, 5, 7, 11, 21, 41]
rows = []
for k in K_GRID:
    rows.append(run_experiment(f'kNN, k={k}', ..., ..., ..., True, k))
k_df = pd.DataFrame(rows)
display(k_df)

plt.figure(figsize=(6.5, 4))
...
plt.axhline(0.5, color='gray', linestyle='--', label='порог 0,5')
plt.xlabel('k'); plt.ylabel('метрика на validation'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

# правило выбора k
candidates = k_df[k_df['val_precision_1'] >= 0.5]
best_row = ...
best_k = int(best_row['k'])
print('Выбрано k =', best_k)

**Ответ:**
1. Что происходит с recall и precision при росте `k`? Объясните через идею «голосования соседей» и долю класса 1 в данных.
2. Почему правило выбора `k` нужно было зафиксировать **до** просмотра таблицы?

## Задание 7. Проверка критерия успеха на validation (шаг 7)

Для выбранного `k` посчитайте recall и precision на validation и сравните с критерием из задания 0 (recall ≥ 0,50 **и** precision ≥ 0,50). Выведите строку вида: «критерий выполнен» / «критерий НЕ выполнен».

Затем ответьте письменно: **что делать, если критерий не выполнен?** Перечислите допустимые и недопустимые действия.

In [ ]:
final_model = Pipeline([
    ('prep', make_preprocessor(NUM_COLS, ORD_COLS, NOM_COLS, scale=True)),
    ('knn', KNeighborsClassifier(n_neighbors=best_k)),
])
final_model.fit(X_train, y_train)
val_pred = final_model.predict(X_val)

val_recall = ...
val_precision = ...
print(f'k={best_k}: recall={val_recall:.3f}, precision={val_precision:.3f}')
print('Критерий', ...)

**Ответ:** Что делать, если критерий не выполнен?
- допустимо: …
- недопустимо: …

## Задание 8. Анализ ошибок на validation (шаг 8)

1. Постройте confusion matrix для `final_model` на validation с подписями строк и столбцов (класс 1 = «уйдёт»). Выпишите TP, FN, FP, TN.
2. Посчитайте **recall класса 1 внутри каждой группы** по `contract` (на validation). Для каких клиентов модель чаще всего пропускает уходящих?

Не забудьте: доли внутри группы **не** складываются в 1.

In [ ]:
cm = confusion_matrix(y_val, val_pred, labels=[1, 0])
cm_df = pd.DataFrame(cm,
    index=['истина 1 (ушёл)', 'истина 0 (остался)'],
    columns=['прогноз 1', 'прогноз 0'])
display(cm_df)
tp, fn, fp, tn = ...
print('TP, FN, FP, TN:', tp, fn, fp, tn)

# recall по группам contract
err = pd.DataFrame({'contract': X_val['contract'].to_numpy(), 'y': y_val.to_numpy(), 'pred': val_pred})
...

**Ответ:**
1. Что означает FN в терминах бизнеса? Какая ошибка дороже и совпадает ли это с выбранной метрикой?
2. В какой группе по `contract` модель хуже всего находит уходящих? Предложите **гипотезу** (не вывод!) для следующего цикла эксперимента.
3. Можно ли по результатам анализа ошибок на validation теперь «подкрутить» модель и снова смотреть validation? Что в таком случае происходит с validation как с «чистым» набором?

## Задание 9. Финальная проверка на test — один раз (шаг 9)

Конфигурация уже выбрана: признаки из задания 6, `k = best_k`, обучение на train. Теперь **один раз** вычислите на test accuracy, recall и precision класса 1 и сравните с критерием.

⚠️ После этой ячейки её содержимое **не меняем**: ни `k`, ни признаки, ни порог. Результат фиксируем как есть.

In [ ]:
test_pred = final_model.predict(...)
print('Test accuracy :', ...)
print('Test recall   :', ...)
print('Test precision:', ...)

**Ответ (вывод по проекту, 5–8 предложений):** Что вы выбрали и почему, каковы метрики на validation и test, выполнен ли критерий, как вы объясните разницу между validation и test, можно ли запускать модель в работу и что нужно мониторить после запуска.

## Задание 10. Найди нарушение workflow

Ниже 5 фрагментов «чужого» кода. Для каждого укажите: **какое правило нарушено**, **к какой утечке/ошибке это приводит** и **как исправить**. Код запускать не нужно.

**Фрагмент A**
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)                  # X — вся таблица
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y)
```

**Фрагмент B**
```python
X = pd.get_dummies(df.drop(columns='churn'))        # кодирование до split
X = X.fillna(X.mean())                              # среднее по всей таблице
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y)
```

**Фрагмент C**
```python
best = None
for k in [1, 3, 5, 11, 21]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    score = recall_score(y_test, knn.predict(X_test), pos_label=1)
    if best is None or score > best[1]:
        best = (k, score)
print('Честная оценка на test:', best[1])
```

**Фрагмент D**
```python
X = df.drop(columns=['churn', 'customer_id'])       # cancel_reason остаётся в признаках
```

**Фрагмент E**
```python
test_recall = recall_score(y_test, final_model.predict(X_test), pos_label=1)
# 0.44 — меньше критерия. Добавим признак «звонки в поддержку на месяц стажа» и перепроверим test
X_test['calls_per_month'] = X_test['support_calls'] / X_test['tenure_months']
...
```

**Ответ:**

| Фрагмент | Нарушенное правило | Чем плохо | Как исправить |
|---|---|---|---|
| A | | | |
| B | | | |
| C | | | |
| D | | | |
| E | | | |

## Задание 11 ★ (по желанию). Эксперимент-диверсия: насколько утечка «улучшает» метрики

Добавьте в признаки **`cancel_reason`** (как номинальный, one-hot), обучите kNN с тем же `k` и сравните recall/precision на validation с честной моделью. Соберите свой журнал: честная модель vs «модель с утечкой».

Объясните, почему такой результат **нельзя** выдавать заказчику, и почему он не воспроизведётся в реальной работе модели.

In [ ]:
# Подсказка: нужны X_train_leak / X_val_leak — те же строки, но с колонкой cancel_reason.
# Индексы train / val берём из уже готовых X_train и X_val, чтобы split остался прежним.
leak_train = df.loc[X_train.index, ['cancel_reason']]
...

**Ответ:** Почему метрики «с утечкой» вводят в заблуждение?

## Чек-лист самопроверки перед сдачей

1. Критерий успеха записан **до** экспериментов и не менялся?
2. `customer_id` и `cancel_reason` не попали в признаки, и вы можете объяснить почему?
3. Split сделан **до** любых `fit`, со `stratify` и `random_state`?
4. Импутер, энкодер и scaler обучены **только на train**?
5. `handle_unknown='ignore'` стоит, порядок категорий для ordinal задан явно?
6. Baseline посчитан и присутствует в журнале?
7. В журнале одна гипотеза за раз, один split, одни и те же метрики?
8. `k` выбран по **validation** по правилу, зафиксированному заранее?
9. Test использован **ровно один раз**, после выбора конфигурации?
10. Ноутбук проходит `Restart & Run All`?

## Оценивание (20 баллов + 2 бонус)

| Блок | Баллы |
|---|---|
| Задание 0: постановка и критерий | 2 |
| Задание 1: аудит и обоснование исключённых колонок | 3 |
| Задание 2: корректный split | 2 |
| Задание 3: baseline и вывод | 1 |
| Задание 4: pipeline без утечек | 3 |
| Задание 5: журнал экспериментов | 3 |
| Задание 6–7: подбор `k`, проверка критерия | 2 |
| Задание 8: анализ ошибок | 1 |
| Задание 9: один раз test + вывод | 1 |
| Задание 10: поиск нарушений | 2 |
| Задание 11 ★ | +2 |

**Штрафы:** использование test до задания 9 — −5 баллов; `fit` на validation/test — −3 за каждое нарушение; `cancel_reason` в итоговой модели — −5.